#**2nd Week**

##**Задачи - Подзапросы (Продвинутый уровень)**

В этом тесте вам предстоит решить практические задачи продвинутого уровня на тему "Подзапросы".

Найти первые 50 идентификаторов клиентов, которые бронировали номера на самый долгий срок, отсортировав их по возрастанию. Идентификаторы должны быть уникальными.

In [ ]:
SELECT DISTINCT renter_id
FROM (
    SELECT 
        renter_id,
        (julianday(check_out_date) - julianday(check_in_date)) AS duration
    FROM 
        bookings
) AS booking_durations
ORDER BY 
    duration DESC  -- Sort by duration in descending order
LIMIT 50;  -- Limit to the top 50 longest durations

Найдите все номера (room_number), которые стоят дороже любого номера типа "Стандартный" (Для этого нужно в подзапросе определить максимальную цену для номеров этого типа).

In [ ]:
SELECT room_number
FROM rooms
WHERE price_per_night > (
    SELECT MAX(price_per_night)
    FROM rooms
    WHERE type_name = 'Стандартный'
);


Выведите ID первых 50 бронирований, в которых клиенты оставили наивысшую оценку (Для этого определите максимально возможную оценку в подзапросе). Отсортируйте вывод по id бронирования.

In [ ]:
SELECT booking_id
FROM bookings
WHERE booking_id IN (
    SELECT booking_id
    FROM ratings
    WHERE rating_value = (
        SELECT MAX(rating_value)
        FROM ratings
    )
)
ORDER BY booking_id
LIMIT 50;


Найти номера (room_number), которые были забронированы клиентами, оплатившими бронирование с помощью 'СБП', но только для номеров типа 'Люкс'. Отсортируйте их по возрастанию.
 
Рекомендация: можно использовать подзапросы в нескольких условиях внутри WHERE, объединенными AND.

In [ ]:
SELECT DISTINCT r.room_number
FROM bookings b
JOIN rooms r ON b.room_number = r.room_number  -- Join bookings with rooms to access room types
JOIN payments p ON b.booking_id = p.booking_id  -- Join with payments to access payment methods
WHERE p.payment_method = 'СБП'  -- Filter for payment method
  AND r.type_name = 'Люкс'  -- Filter for room type
ORDER BY r.room_number;  -- Sort by room number in ascending order

Выведите номера (room_number) и их ранг (room_rank), площадь которых входит в топ-3 самых больших площадей номеров, используя функцию DENSE_RANK. В результате может быть отображено более 3 записей, если несколько комнат имеют одинаковую площадь, которая входит в топ-3. Отсортируйте результат по рангу и номеру комнаты.

In [ ]:
WITH RankedRooms AS (
    SELECT 
        room_number,
        room_size_sqm,
        DENSE_RANK() OVER (ORDER BY room_size_sqm DESC) AS room_rank
    FROM 
        rooms
)

SELECT 
    room_number,
    room_rank
FROM 
    RankedRooms
WHERE 
    room_rank <= 3  -- Filter for top 3 ranks
ORDER BY 
    room_rank, room_number;  -- Sort by rank and room number

Выведите номера (room_number) и их ранг (booking_rank), которые находятся в топ-10 по количеству бронирований, используя функцию DENSE_RANK, и отсортируйте их по рангу и номеру комнаты. В результате может быть отображено более 10 записей, если несколько комнат имеют одинаковое количество бронирований, которое входит в топ-10.

In [ ]:
WITH BookingCounts AS (
    SELECT 
        room_number,
        COUNT(*) AS booking_count
    FROM 
        bookings
    GROUP BY 
        room_number
),
RankedRooms AS (
    SELECT 
        room_number,
        booking_count,
        DENSE_RANK() OVER (ORDER BY booking_count DESC) AS booking_rank
    FROM 
        BookingCounts
)

SELECT 
    room_number,
    booking_rank
FROM 
    RankedRooms
WHERE 
    booking_rank <= 10  -- Filter for top 10 ranks
ORDER BY 
    booking_rank, room_number;  -- Sort by rank and room number